In [2]:

import pandas as pd
import numpy as np
import re
import os
import json
from tqdm import tqdm
from hazm import Normalizer, word_tokenize
from sklearn.model_selection import train_test_split

tqdm.pandas()

# =========================================
# 1. PATHS
# =========================================

BASE_PATH = r"C:\Users\Neda\Desktop\personality_llm"

RAW_DATA_PATH = os.path.join(BASE_PATH, "data", "raw")
PROCESSED_DATA_PATH = os.path.join(BASE_PATH, "data", "processed")

TRAIN_PATH = os.path.join(RAW_DATA_PATH, "train.csv")
TEST_PATH = os.path.join(RAW_DATA_PATH, "test.csv")
KAMTERA_PATH = os.path.join(RAW_DATA_PATH, "Kamtera.json")

os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

print("RAW_PATH:", RAW_DATA_PATH)
print("PROCESSED_PATH:", PROCESSED_DATA_PATH)

# =========================================
# 2. NORMALIZER
# =========================================

normalizer = Normalizer()

# =========================================
# 3. CLEANING FUNCTION (IMPROVED)
# =========================================

def clean_persian_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Normalize Arabic -> Persian
    text = text.replace("ي", "ی")
    text = text.replace("ك", "ک")
    text = text.replace("ة", "ه")
    text = text.replace("ۀ", "ه")

    # Remove URLs, emails, mentions
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)

    # Remove hashtags
    text = text.replace("#", " ")

    # Remove English + numbers
    text = re.sub(r"[A-Za-z]", " ", text)
    text = re.sub(r"[0-9۰-۹]", " ", text)

    # Keep only Persian + basic punctuation
    text = re.sub(r"[^\u0600-\u06FF\s\.\!\؟\?،؛]", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # Final normalization (hazm)
    text = normalizer.normalize(text)

    return text


# =========================================
# 4. FEATURE ENGINEERING (IMPROVED)
# =========================================

def extract_style_features(text):
    text = str(text)
    tokens = text.split()

    if len(tokens) == 0:
        return pd.Series({
            "num_words": 0,
            "num_chars": 0,
            "avg_word_length": 0,
            "type_token_ratio": 0,
            "num_questions": 0,
            "num_exclamations": 0,
            "num_commas": 0,
            "num_sentences": 0
        })

    unique_tokens = set(tokens)

    return pd.Series({
        "num_words": len(tokens),
        "num_chars": len(text),
        "avg_word_length": np.mean([len(w) for w in tokens]),
        "type_token_ratio": len(unique_tokens) / len(tokens),
        "num_questions": text.count("?") + text.count("؟"),
        "num_exclamations": text.count("!"),
        "num_commas": text.count("،"),
        "num_sentences": len(re.split(r"[.!؟?]", text))
    })


def safe_tokenize(text):
    try:
        return word_tokenize(str(text))
    except:
        return []


# =========================================
# 5. LOAD DATA
# =========================================

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH, header=None, names=["text"])

train_df.columns = train_df.columns.str.strip().str.lower()
train_df = train_df.rename(columns={"comment": "text", "personality": "label"})

print("Train:", train_df.shape)
print("Test:", test_df.shape)

# =========================================
# 6. PREPROCESS PIPELINE (TRAIN + TEST)
# =========================================

def preprocess_df(df, is_train=True):
    df = df.copy()

    df["clean_text"] = df["text"].progress_apply(clean_persian_text)

    df = df[df["clean_text"].str.split().str.len() >= 3].reset_index(drop=True)

    features = df["clean_text"].progress_apply(extract_style_features)
    df = pd.concat([df, features], axis=1)

    df["tokens"] = df["clean_text"].progress_apply(safe_tokenize)

    return df


train_df = preprocess_df(train_df, is_train=True)
test_df = preprocess_df(test_df, is_train=False)

# =========================================
# 7. TRAIN / VALIDATION SPLIT (IMPORTANT FIX)
# =========================================

train_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df["label"]
)

print("\nTrain:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

# =========================================
# 8. SAVE CLEAN DATA
# =========================================

train_df.to_csv(os.path.join(PROCESSED_DATA_PATH, "train_clean.csv"), index=False, encoding="utf-8-sig")
val_df.to_csv(os.path.join(PROCESSED_DATA_PATH, "val_clean.csv"), index=False, encoding="utf-8-sig")
test_df.to_csv(os.path.join(PROCESSED_DATA_PATH, "test_clean.csv"), index=False, encoding="utf-8-sig")

print("\nSaved Kaggle datasets.")

# =========================================
# 9. KAMTERA PROCESSING (IMPROVED SAFE VERSION)
# =========================================

def read_kamtera(path):
    try:
        return pd.read_json(path)
    except:
        try:
            return pd.read_json(path, lines=True)
        except:
            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)
            return pd.DataFrame(data)


if os.path.exists(KAMTERA_PATH):

    kamtera_df = read_kamtera(KAMTERA_PATH)
    kamtera_df.columns = kamtera_df.columns.astype(str).str.lower()

    text_col = kamtera_df.select_dtypes(include=["object"]).columns[0]
    kamtera_df = kamtera_df.rename(columns={text_col: "text"})

    kamtera_df = kamtera_df[["text"]].copy()

    print("Kamtera shape:", kamtera_df.shape)

    kamtera_df["clean_text"] = kamtera_df["text"].progress_apply(clean_persian_text)

    kamtera_df = kamtera_df[kamtera_df["clean_text"].str.split().str.len() >= 3]

    features = kamtera_df["clean_text"].progress_apply(extract_style_features)
    kamtera_df = pd.concat([kamtera_df, features], axis=1)

    kamtera_df["tokens"] = kamtera_df["clean_text"].progress_apply(safe_tokenize)

    kamtera_df.to_csv(
        os.path.join(PROCESSED_DATA_PATH, "kamtera_clean.csv"),
        index=False,
        encoding="utf-8-sig"
    )

    print("Kamtera saved.")

else:
    print("Kamtera not found")

# =========================================
# 10. FINAL SUMMARY
# =========================================

print("\n====================")
print("FINAL SUMMARY")
print("====================")

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

if os.path.exists(KAMTERA_PATH):
    print("Kamtera:", kamtera_df.shape)

print("\nDONE ✔")

RAW_PATH: C:\Users\Neda\Desktop\personality_llm\data\raw
PROCESSED_PATH: C:\Users\Neda\Desktop\personality_llm\data\processed
Train: (1000, 2)
Test: (800, 1)


100%|██████████| 800/800 [00:00<00:00, 50273.33it/s]



Train: (800, 12)
Validation: (200, 12)
Test: (800, 11)

Saved Kaggle datasets.
Kamtera shape: (170234, 1)


100%|██████████| 107280/107280 [00:01<00:00, 85359.65it/s]


Kamtera saved.

FINAL SUMMARY
Train: (800, 12)
Val: (200, 12)
Test: (800, 11)
Kamtera: (107280, 11)

DONE ✔
